### Unsafe Parallel Code

In [28]:
import numpy as np
import threading
import time
from PIL import Image

In [ ]:
image_path = "test_images/large_1024x1024.png"
image = np.array(Image.open(image_path).convert("L"), dtype=np.float64)

In [30]:
gaussian_kernel = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
], dtype=np.float64) / 16

In [ ]:
# Race condition 1
def pixel_worker(shared_output, partial_sum, repeats):

    for _ in range(repeats):

        temp = shared_output[0]
        time.sleep(0)            # for thread switching
        shared_output[0] = temp + partial_sum


def test_output_pixel(n_workers=3):

    # center pixel
    i, j = image.shape[0] // 2, image.shape[1] // 2

    partial_sums = []

    for k in range(n_workers):

        partial = 0

        for l in range(3):
            partial += (
                image[i + k - 1, j + l - 1]
                * gaussian_kernel[k, l]
            )

        partial_sums.append(partial)

    shared_output = [0.0]
    repeats = 1000

    threads = [
        threading.Thread(
            target=pixel_worker,
            args=(shared_output, partial, repeats)
        )
        for partial in partial_sums
    ]

    start = time.perf_counter()

    for t in threads:
        t.start()

    for t in threads:
        t.join()

    elapsed = time.perf_counter() - start

    expected = sum(partial_sums) * repeats

    print("Expected:", expected)
    print("Actual:", shared_output[0])
    print("Absolute error:", abs(expected - shared_output[0]))
    print(f"Execution time: {elapsed:.4f}")

In [40]:
for run in range(5):

    print(f"\n RUN {run + 1}")
    test_output_pixel()


 RUN 1
Expected: 18062.5
Actual: 5144.0625
Absolute error: 12918.4375
Execution time: 0.1671

 RUN 2


Expected: 18062.5
Actual: 9125.0
Absolute error: 8937.5
Execution time: 0.1659

 RUN 3
Expected: 18062.5
Actual: 5150.875
Absolute error: 12911.625
Execution time: 0.1701

 RUN 4
Expected: 18062.5
Actual: 9128.8125
Absolute error: 8933.6875
Execution time: 0.1563

 RUN 5
Expected: 18062.5
Actual: 9128.8125
Absolute error: 8933.6875
Execution time: 0.1101


In [43]:
# Race condition 2
def counter_worker(counter, iterations):

    for _ in range(iterations):

        temp = counter[0]
        time.sleep(0)  
        counter[0] = temp + 1


def test_counter(n_workers=2):

    counter = [0]
    iterations = 50000

    threads = [
        threading.Thread(
            target=counter_worker,
            args=(counter, iterations)
        )
        for _ in range(n_workers)
    ]

    start = time.perf_counter()

    for t in threads:
        t.start()

    for t in threads:
        t.join()

    elapsed = time.perf_counter() - start

    expected = n_workers * iterations

    print("Expected:", expected)
    print("Actual:", counter[0])
    print("Lost updates:", expected - counter[0])
    print(f"Execution time: {elapsed:.4f}")

In [44]:
for run in range(5):

    print(f"\n RUN {run + 1}")
    test_counter()


 RUN 1
Expected: 100000
Actual: 50000
Lost updates: 50000
Execution time: 5.8033

 RUN 2
Expected: 100000
Actual: 50001
Lost updates: 49999
Execution time: 5.8746

 RUN 3
Expected: 100000
Actual: 50002
Lost updates: 49998
Execution time: 5.3925

 RUN 4
Expected: 100000
Actual: 50002
Lost updates: 49998
Execution time: 5.9272

 RUN 5
Expected: 100000
Actual: 50000
Lost updates: 50000
Execution time: 5.7691
